# Four-Method Common-Timestamp Comparison

Please cite the following paper when you use or reference this code
Oluwafolajimi Samuel Bolusteve, Linhan Fang, and Xingpeng Li, “Net Load Forecasting with Renewable Capacity Features: A Comparison of Direct and Indirect Neural Network Methods”, IEEE Texas Power and Energy Conference, College Station, TX, USA, Feb. 2027.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def find_project_root():
    """Find the repository root when Jupyter is launched from inside the repository."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "Data Download").is_dir() and (candidate / "FCNN Direct Method").is_dir():
            return candidate
    raise FileNotFoundError(
        "Cannot locate the repository root containing 'Data Download' and the method folders."
    )


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "Data Download"
FIXED_DATA_DIR = PROJECT_ROOT / "fixed_data"
DATA_FILE = FIXED_DATA_DIR / "ercot_2021_2024_hourly.csv"
OUTPUT_DIR = PROJECT_ROOT / "fixed_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TIME_COLUMN = "time"
files = {
    "FCNN Direct": "fcnn_direct_net_load_predictions.xlsx",
    "FCNN Indirect": "fcnn_indirect_net_load_predictions.xlsx",
    "LSTM Direct": "lstm_direct_net_load_predictions.xlsx",
    "LSTM Indirect": "lstm_indirect_net_load_predictions.xlsx",
}


In [2]:
frames = {}
for method, filename in files.items():
    frame = pd.read_excel(OUTPUT_DIR / filename)
    required = {"time", "actual", "predict"}
    if not required.issubset(frame.columns):
        raise KeyError(f"{method} is missing {sorted(required.difference(frame.columns))}")
    frame = frame[["time", "actual", "predict"]].copy()
    frame["time"] = pd.to_datetime(frame["time"], errors="raise")
    if frame["time"].duplicated().any():
        raise ValueError(f"{method} contains duplicate timestamps.")
    frames[method] = frame.sort_values("time").reset_index(drop=True)

common_times = set.intersection(*(set(frame["time"]) for frame in frames.values()))
if not common_times:
    raise ValueError("The four methods have no common test timestamps.")
common_times = pd.DatetimeIndex(sorted(common_times))
for method, frame in frames.items():
    print(method, "original rows:", len(frame), "common rows:", len(common_times))


FCNN Direct original rows: 1754 common rows: 1754
FCNN Indirect original rows: 1754 common rows: 1754
LSTM Direct original rows: 1754 common rows: 1754
LSTM Indirect original rows: 1754 common rows: 1754


In [3]:
aligned = {}
for method, frame in frames.items():
    aligned[method] = (
        pd.DataFrame({"time": common_times})
        .merge(frame, on="time", how="left", validate="one_to_one")
    )

reference_actual = aligned["FCNN Direct"]["actual"].to_numpy(dtype=float)
for method, frame in aligned.items():
    actual = frame["actual"].to_numpy(dtype=float)
    if not np.allclose(actual, reference_actual, rtol=1e-9, atol=1e-6):
        raise ValueError(
            f"{method} has a different actual net-load target; "
            f"max difference={np.max(np.abs(actual-reference_actual)):.6f} MW"
        )


In [4]:
rows = []
for method, frame in aligned.items():
    actual = frame["actual"].to_numpy(dtype=float)
    predicted = frame["predict"].to_numpy(dtype=float)
    error = predicted - actual
    mse = mean_squared_error(actual, predicted)
    variance = np.var(actual)
    denominator = np.abs(actual) + np.abs(predicted)
    smape = np.divide(
        2.0*np.abs(error), denominator,
        out=np.zeros_like(denominator), where=denominator > 1e-9,
    )
    mask = np.abs(actual) > 1.0
    rows.append({
        "method": method,
        "sample_count": len(actual),
        "mae_mw": mean_absolute_error(actual, predicted),
        "rmse_mw": np.sqrt(mse),
        "nmse": mse/variance if variance > 1e-12 else np.nan,
        "r2": r2_score(actual, predicted),
        "smape_percent": np.mean(smape)*100.0,
        "masked_mape_percent": np.mean(np.abs(error[mask]/actual[mask]))*100.0,
    })

comparison = pd.DataFrame(rows).sort_values("rmse_mw").reset_index(drop=True)
comparison.to_csv(OUTPUT_DIR / "four_method_common_timestamp_comparison.csv", index=False)
comparison.to_excel(OUTPUT_DIR / "four_method_common_timestamp_comparison.xlsx", index=False)
comparison


,method,sample_count,mae_mw,rmse_mw,nmse,r2,smape_percent,masked_mape_percent
0,LSTM Indirect,1754,695.457324,938.098414,0.012911,0.987089,2.484051,2.463528
1,LSTM Direct,1754,803.934371,1127.733023,0.018658,0.981342,2.739086,2.729514
2,FCNN Direct,1754,1873.519238,2290.754523,0.076985,0.923015,6.822229,7.138111
3,FCNN Indirect,1754,1999.495674,2469.796331,0.089490,0.910510,7.134610,7.397811
